In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import sys
from pathlib import Path
import glob

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

In [2]:
# =========================================================
# Matplotlib style
# =========================================================

params = {
    "ytick.color": "black",
    "xtick.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Serif"],
}

plt.rcParams.update(params)

# Experimental conditions

In [85]:
silver_path = Path.cwd().parent / 'data/silver_layer/'

silver_confocal_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'
gold_profile_path = Path.cwd().parent / 'data/gold_layer/kde_plots'

In [86]:
df_confocal = pd.read_pickle(silver_path / 'confocal_results.pkl')
df_experimental = pd.read_pickle(silver_path / 'experimental_results.pkl')

In [87]:
missing_threshold = 50
list_runs = df_confocal.loc[df_confocal['t2_missing_percentage']<missing_threshold, 'run_id'].unique()
df_selected = df_experimental.loc[df_experimental['run_id'].isin(list_runs)]

In [ ]:
pattern_runs = {

    pattern: group["run_id"].tolist()

    for pattern, group in df_selected.groupby("reduced_pattern")
}

In [89]:
def load_confocal_run(run_id, base_path=silver_confocal_path):
    """
    Load confocal run CSV.

    Expected columns:
        time, t1, t2, t3
    """

    file_path = base_path / f'confocal_run_{run_id}.csv'

    df = pd.read_csv(file_path)

    return df


In [90]:
CM = 1 / 2.54

FIG_WIDTH_CM = 8.4
FIG_HEIGHT_CM = 6.8

FIG_SIZE = (
    FIG_WIDTH_CM * CM,
    FIG_HEIGHT_CM * CM
)

In [185]:
# =========================================================
# KDE plot function
# =========================================================

def kde_pattern_plot(
    pattern,
    run_ids,
    df_experimental,
):

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    palette = sns.color_palette("muted", len(run_ids)) # "deep"

    # -----------------------------------------------------
    # Loop through runs
    # -----------------------------------------------------

    for i, run_id in enumerate(run_ids):

        # Load confocal run
        df_run = load_confocal_run(run_id)

        # Remove NaNs
        df_run = df_run.loc[~df_run["t2"].isna()].copy()

        # Cap film thickness
        df_run.loc[df_run["t2"] > 8.02, "t2"] = 8.02

        # Experimental condition metadata
        row = df_experimental.loc[
            df_experimental["run_id"] == run_id
        ].iloc[0]

        jl = row["jl"]
        jg = row["jg"]

        # KDE plot
        sns.kdeplot(
            x=df_run["t2"],
            ax=ax,
            color=palette[i],
            linewidth=1.0, # 1.2,
            label=rf"{jl:.2f}, {jg:.2f}",
            zorder=2
        )

    # -----------------------------------------------------
    # Axes
    # -----------------------------------------------------

    ax.set_xlim(0, 10)

    ax.set_xlabel("Film thickness [mm]")
    ax.set_ylabel("Density")

    ax.yaxis.set_major_locator(MaxNLocator(nbins=8))

    ax.grid(alpha=0.3, zorder=0)

    # -----------------------------------------------------
    # Legend
    # -----------------------------------------------------
    # ax.set_title(pattern.replace("_", " ").title())

    ax.legend(
        title=r"$j_l$ [m/s], $j_g$ [m/s]",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.98),
        frameon=False,
        ncol=1,
        fontsize=7,
        title_fontsize=7,
    )

    fig.tight_layout()

    plt.close(fig)

    return fig, ax

In [182]:
new_runs = {
    'bubbly': [65, 61, 60, 59, 50, 53],
    'dispersed': [81, 80, 74, 75, 76, 77],
    'plug': [35, 33, 36, 31, 28, 66],
    'slug': [15, 14, 13, 25, 87, 56],
    'wavy': [40, 41, 148, 47, 46, 44]
}

In [ ]:
# =========================================================
# Generate all plots
# =========================================================
figures = {}

# for pattern, run_ids in pattern_runs.items():
for pattern, run_ids in new_runs.items():

    fig, ax = kde_pattern_plot(
        pattern=pattern,
        run_ids=run_ids,
        df_experimental=df_selected
    )

    figures[pattern] = fig

    current_fig_path = gold_profile_path / f'kde_{pattern}.pdf'

    fig.savefig(
        current_fig_path,
        format='pdf',
        bbox_inches='tight'
    )

    plt.close(fig)
